# Create new Ressource file

All Ressources 

In [3]:
import pandas as pd
from pathlib import Path

BASE = Path("/Users/simonimmler/PycharmProjects/Praktikum/bppso-groupwork")   # Repo-Root anpassen
src = BASE / "resources" / "availabilities" / "availabilities_advanced.csv"
dst = BASE / "resources" / "availabilities" / "availabilities_9to5.csv"

df = pd.read_csv(src)

# gleiche Ressourcen + gleiche DayIds behalten
df["StartTime"] = "09:00:00"
df["EndTime"]   = "17:00:00"

# Breaks konsistent halten
if "DurationMin" in df.columns:
    df["DurationMin"] = df["DurationMin"].fillna(0).astype(int)
    if "BreakStart" in df.columns:
        has_break = df["DurationMin"] > 0
        df.loc[has_break, "BreakStart"] = "13:00:00"
        df.loc[~has_break, "BreakStart"] = "00:00:00"

df.to_csv(dst, index=False)
print("Saved:", dst)
print(df.head())

Saved: /Users/simonimmler/PycharmProjects/Praktikum/bppso-groupwork/resources/availabilities/availabilities_9to5.csv
  Resource  DayId StartTime   EndTime BreakStart  DurationMin
0   User_1      0  09:00:00  17:00:00   00:00:00            0
1   User_1      1  09:00:00  17:00:00   00:00:00            0
2   User_1      2  09:00:00  17:00:00   00:00:00            0
3   User_1      3  09:00:00  17:00:00   00:00:00            0
4   User_1      4  09:00:00  17:00:00   00:00:00            0


In [18]:
%%capture
import pandas as pd
%run ./evaluation.ipynb

In [21]:
log_9to5 = pd.read_csv(BASE / "simulation_evaluation" /"results"/ "sim_output_9to5.csv",
                       parse_dates=["time:timestamp"])

avail_9to5_df = pd.read_csv(BASE / "resources" / "availabilities" / "availabilities_9to5.csv")
avail_9to5_df["StartTime"]= pd.to_datetime(avail_9to5_df["StartTime"], format="%H:%M:%S").dt.time
avail_9to5_df["EndTime"]= pd.to_datetime(avail_9to5_df["EndTime"], format="%H:%M:%S").dt.time
avail_9to5_df["DurationMin"] = avail_9to5_df["DurationMin"].fillna(0).astype(int)

# completed cases only
log_9to5_completed = log_9to5[log_9to5["case:concept:name"].isin(
    log_9to5.groupby("case:concept:name").size()[lambda s: s > 2].index
)]

ct_9to5 = compute_cycle_times(log_9to5_completed)
work_9to5 = compute_working_seconds(log_9to5)
avail_9to5 = compute_available_seconds(avail_9to5_df)
occ_9to5 = compute_occupation(work_9to5, avail_9to5)
_, mad_9to5, wmad_9to5 = fairness_metrics(occ_9to5["occupation"], occ_9to5["available_seconds"])

compare_9to5 = pd.DataFrame({
    "Metric": [
        "# Completed Cases",
        "Avg Cycle Time (h)",
        "Avg Resource Occup (%)",
        "Fairness MAD (unwt)",
        "Fairness MAD (wtd)",
    ],
    "Advanced baseline": [
        len(advanced_ct),
        round(advanced_ct.mean(), 4),
        round(advanced_occ["occupation"].mean() * 100, 4),
        round(a_mad, 5),
        round(a_wmad, 5),
    ],
    "9-to-5": [
        len(ct_9to5),
        round(ct_9to5.mean(), 4),
        round(occ_9to5["occupation"].mean() * 100, 4),
        round(mad_9to5, 5),
        round(wmad_9to5, 5),
    ],
})

compare_9to5["Delta"] = compare_9to5["9-to-5"] - compare_9to5["Advanced baseline"]
print(compare_9to5.to_string(index=False))

                Metric  Advanced baseline    9-to-5      Delta
     # Completed Cases          413.00000 219.00000 -194.00000
    Avg Cycle Time (h)            2.49750   2.30610   -0.19140
Avg Resource Occup (%)            1.63720   0.30190   -1.33530
   Fairness MAD (unwt)            0.02689   0.00248   -0.02441
    Fairness MAD (wtd)            0.03073   0.00250   -0.02823


Impact of 9-to-5 schedule

Completed cases ↓ 456 → 417

Cycle time ↑ 2.46h → 3.65h (+48%)

Resource occupation slightly ↓

Fairness ≈ unchanged